## Combien de lignes et de colonnes ? Quels types de données ?

In [4]:
import pandas as pd

df = pd.read_csv("../data/raw/974_2024.csv.gz", low_memory=False)
df.shape, df.info()

<class 'pandas.DataFrame'>
RangeIndex: 19251 entries, 0 to 19250
Data columns (total 40 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   id_mutation                   19251 non-null  str    
 1   date_mutation                 19251 non-null  str    
 2   numero_disposition            19251 non-null  int64  
 3   nature_mutation               19251 non-null  str    
 4   valeur_fonciere               19203 non-null  float64
 5   adresse_numero                16673 non-null  float64
 6   adresse_suffixe               1766 non-null   str    
 7   adresse_nom_voie              19223 non-null  str    
 8   adresse_code_voie             19223 non-null  str    
 9   code_postal                   19223 non-null  float64
 10  code_commune                  19251 non-null  int64  
 11  nom_commune                   19251 non-null  str    
 12  code_departement              19251 non-null  int64  
 13  ancien_code_

((19251, 40), None)

19251 lignes et 40 colonnes. C'est soit des str, soit des int, soit des float.


## Quelles colonnes ont beaucoup de valeurs manquantes ? Pourquoi, à ton avis ?

In [ ]:
# En pourcentage :
print((df.isnull().mean() * 100).sort_values(ascending=False))

lot4_surface_carrez             100.000000
ancien_id_parcelle              100.000000
ancien_code_commune             100.000000
lot5_surface_carrez             100.000000
ancien_nom_commune              100.000000
lot3_surface_carrez              99.989611
lot5_numero                      99.948055
lot2_surface_carrez              99.942860
lot4_numero                      99.844164
numero_volume                    99.553270
lot3_numero                      99.516908
lot1_surface_carrez              98.929926
lot2_numero                      98.644226
nature_culture_speciale          98.633837
code_nature_culture_speciale     98.633837
adresse_suffixe                  90.826451
lot1_numero                      66.266687
surface_reelle_bati              58.158018
surface_terrain                  34.595605
code_nature_culture              34.595605
nature_culture                   34.595605
nombre_pieces_principales        26.808997
type_local                       26.788219
code_type_l

On repere les colonnes comme le numéro d'adresse, le nombre de pieces principales ou meme le type de local. Et on a meme des colonnes qui sont completement vide comme le code de commune, le code du departement ou meme le nombre de lots.

## Quelles valeurs prennent nature_mutation et type_local ?

In [ ]:
print("Pour la colonne 'nature_mutation', voici le nombre de valeurs manquantes et les différentes valeurs présentes :")
print(df['nature_mutation'].value_counts(dropna=False))

print("\nPour la colonne 'type_local', voici le nombre de valeurs manquantes et les différentes valeurs présentes :")
print(df['type_local'].value_counts(dropna=False))

Pour la colonne 'nature_mutation', voici le nombre de valeurs manquantes et les différentes valeurs présentes :
nature_mutation
Vente                                 17542
Vente en l'état futur d'achèvement     1319
Vente terrain à bâtir                   202
Echange                                 118
Adjudication                             70
Name: count, dtype: int64

Pour la colonne 'type_local', voici le nombre de valeurs manquantes et les différentes valeurs présentes :
type_local
Dépendance                                  6003
NaN                                         5157
Appartement                                 4337
Maison                                      3228
Local industriel. commercial ou assimilé     526
Name: count, dtype: int64


nature_mutation prend Vente, Vente en l'état futur d'achèvement, Vente terrain à bâtir, Echange ou                            Adjudication.    
type_local prend   Dépendance, Appartement, Maison, Local industriel. commercial ou assimilé y compris beaucoup de valeurs nulles.   

## Compte les lignes par id_mutation. Pourquoi une même vente apparaît-elle sur plusieurs lignes ? Que se passe-t-il si tu fais un mean() naïf de valeur_fonciere ? (C’est LE piège de DVF, prends le temps de le comprendre.)

In [16]:
# Compter combien de fois chaque id_mutation apparaît
counts = df['id_mutation'].value_counts()
print(counts.head(10))

print("\nExemple de mutation avec plusieurs lignes :")

# Regarder un exemple de mutation qui a plusieurs lignes
# Afficher les lignes pour cette mutation précise
exemple = df[df['id_mutation'] == '2024-1185753']
print(exemple[['id_mutation', 'valeur_fonciere', 'lot1_numero', 'surface_reelle_bati', 'type_local']])

id_mutation
2024-1188637    3048
2024-1187596     303
2024-1186245     220
2024-1189429     122
2024-1187895     103
2024-1186146     102
2024-1189612      85
2024-1189616      79
2024-1192287      72
2024-1185970      69
Name: count, dtype: int64

Exemple de mutation avec plusieurs lignes :
    id_mutation  valeur_fonciere lot1_numero  surface_reelle_bati   type_local
0  2024-1185753         197000.0         144                  NaN   Dépendance
1  2024-1185753         197000.0          21                  NaN   Dépendance
2  2024-1185753         197000.0          18                 65.0  Appartement


Car en France, y'a l'histoire de copropriété ou par exemple une cave compte comme un bien en lui même.

## Quelles valeurs de valeur_fonciere sont aberrantes (1 €, plusieurs dizaines de millions) ?

In [20]:
print(df['valeur_fonciere'].describe())
print(df[df['valeur_fonciere'] <= 1][['date_mutation', 'valeur_fonciere', 'nom_commune']].head())
print(df[df['valeur_fonciere'] >= 10000000][['date_mutation', 'valeur_fonciere', 'nom_commune']].head())

count    1.920300e+04
mean     2.663787e+06
std      4.476859e+06
min      1.000000e+00
25%      1.180000e+05
50%      2.550000e+05
75%      2.223000e+06
max      2.378320e+07
Name: valeur_fonciere, dtype: float64
     date_mutation  valeur_fonciere   nom_commune
290     2024-01-25              1.0   Saint-Denis
2150    2024-03-06              1.0    Saint-Paul
2151    2024-03-06              1.0    Saint-Paul
2494    2024-02-19              1.0  Saint-Pierre
2495    2024-02-19              1.0  Saint-Pierre
     date_mutation  valeur_fonciere    nom_commune
1063    2024-02-13       12871500.0  La Possession
1064    2024-02-13       12871500.0  La Possession
1065    2024-02-13       12871500.0  La Possession
1066    2024-02-13       12871500.0  La Possession
1067    2024-02-13       12871500.0  La Possession


On peut par exemple retrouver des biens à 1 euro : 
290     2024-01-25              1.0   Saint-Denis
2150    2024-03-06              1.0    Saint-Paul
2151    2024-03-06              1.0    Saint-Paul
2494    2024-02-19              1.0  Saint-Pierre
2495    2024-02-19              1.0  Saint-Pierre

Ou bien d'autres qui dépassent des dizaines de millions d'euros : 
1063    2024-02-13       12871500.0  La Possession
1064    2024-02-13       12871500.0  La Possession
1065    2024-02-13       12871500.0  La Possession
1066    2024-02-13       12871500.0  La Possession
1067    2024-02-13       12871500.0  La Possession

## Calcule un prix médian au m² pour les appartements et les maisons vendus seuls, par commune. Quelles communes sont les plus chères ?

In [25]:
df_clean = df.copy()
df_clean['prix_m2'] = df_clean['valeur_fonciere'] / df_clean['surface_reelle_bati']

# # Filtrer pour les APPARTEMENTS
# appart = df_clean[
#     (df_clean['type_local'] == 'Appartement') & 
#     (df_clean['surface_reelle_bati'] > 0) & 
#     (df_clean['valeur_fonciere'] > 0)
# ]
# print("--- Prix médian au m² : APPARTEMENTS ---")
# print(appart.groupby('nom_commune')['prix_m2'].median().sort_values(ascending=False))

# Filtrer pour les MAISONS
maison = df_clean[
    (df_clean['type_local'] == 'Maison') & 
    (df_clean['surface_reelle_bati'] > 0) & 
    (df_clean['valeur_fonciere'] > 0)
]
print("\n--- Prix médian au m² : MAISONS ---")
print(maison.groupby('nom_commune')['prix_m2'].median().sort_values(ascending=False))


--- Prix médian au m² : MAISONS ---
nom_commune
L'Étang-Salé               4933.333333
Les Avirons                4374.295000
Saint-Paul                 3864.304993
Saint-Leu                  3836.657170
La Possession              3658.333333
Entre-Deux                 3606.862745
Saint-Denis                3371.563446
Saint-Pierre               3254.148352
Petite-Île                 2777.777778
Sainte-Marie               2684.210526
Le Tampon                  2676.056338
Les Trois-Bassins          2598.443223
Saint-Joseph               2398.809524
Saint-Louis                2380.952381
Saint-Philippe             2357.230769
Saint-André                2314.814815
Cilaos                     2207.207207
Bras-Panon                 2138.596142
Saint-Benoît               2100.783026
Sainte-Suzanne             2085.714286
La Plaine-des-Palmistes    1903.225806
Salazie                    1892.552283
Le Port                    1885.245902
Sainte-Rose                1709.526593
Name: prix_m2, 

Communes plus chers pour appartement : 
--- Prix médian au m² : APPARTEMENTS ---
nom_commune
Saint-Paul           150197.430556
Sainte-Suzanne       125364.680851
Saint-André          121617.889706

Communes plus chers pour maison : 
--- Prix médian au m² : APPARTEMENTS ---
--- Prix médian au m² : MAISONS ---
nom_commune
L'Étang-Salé               4933.333333
Les Avirons                4374.295000
Saint-Paul                 3864.304993

### Problèmes de qualité identifiés

On voit tres bien que les données ne sont pas propres. On a par exemple le probleme des doublons id_mutation qu'on va devoir gérer.